# Binary Gate Training and Generation Guide

Simple step-by-step notebook for training the palm/non-palm binary gate model and exporting runtime artifacts.

Generated on: 2026-04-14  
Project: Palm Fruit Ripeness Classification System  
Target output: `saved_models/palm_presence_binary.tflite`

## Section 1 - Pipeline Overview

- Step 1: Prepare binary dataset folders (Palm vs Non-palm).
- Step 2: Validate class folders and image counts.
- Step 3: Run `scripts/train_binary_gate.py` with class-name mapping.
- Step 4: Export model artifacts (`.h5`, FP32 TFLite, INT8 TFLite).
- Step 5: Create canonical runtime model (`palm_presence_binary.tflite`).
- Step 6: Generate manifest with recommended threshold.
- Step 7: Apply threshold and model path in API/CLI runtime settings.

## Section 2 - Configure Paths and Class Names

Use exact class folder names from your dataset.

In [6]:
from pathlib import Path

DATA_DIR = Path(r"C:\Users\jeffy\Documents\PSM\PalmDetector\BinaryGateSplit_80_20")
NON_PALM_CLASS = "Non-palm"
PALM_CLASS = "Palm"

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "scripts" / "train_binary_gate.py").exists():
    candidate = PROJECT_ROOT.parent
    if (candidate / "scripts" / "train_binary_gate.py").exists():
        PROJECT_ROOT = candidate

OUTPUT_DIR = PROJECT_ROOT / "saved_models"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("PROJECT_ROOT:", PROJECT_ROOT.resolve())
print("DATA_DIR:", DATA_DIR)
print("OUTPUT_DIR:", OUTPUT_DIR.resolve())
print("CLASSES:", NON_PALM_CLASS, "/", PALM_CLASS)

PROJECT_ROOT: C:\Users\jeffy\Documents\PSM\Project Coding\PSM_PalmFruitRipenessClassificationSystem
DATA_DIR: C:\Users\jeffy\Documents\PSM\PalmDetector\BinaryGateSplit_80_20
OUTPUT_DIR: C:\Users\jeffy\Documents\PSM\Project Coding\PSM_PalmFruitRipenessClassificationSystem\saved_models
CLASSES: Non-palm / Palm


## Section 3 - Validate Dataset Structure and Counts

In [2]:
def count_images(folder: Path) -> int:
    exts = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
    return sum(1 for p in folder.rglob("*") if p.is_file() and p.suffix.lower() in exts)

paths = {
    "train_non_palm": DATA_DIR / "train" / NON_PALM_CLASS,
    "train_palm": DATA_DIR / "train" / PALM_CLASS,
    "val_non_palm": DATA_DIR / "val" / NON_PALM_CLASS,
    "val_palm": DATA_DIR / "val" / PALM_CLASS,
}

for name, path in paths.items():
    print(f"{name:15} exists={path.exists()} count={count_images(path) if path.exists() else 0}")

train_non_palm  exists=True count=800
train_palm      exists=True count=800
val_non_palm    exists=True count=200
val_palm        exists=True count=200


## Section 4 - Train and Export Binary Gate

This runs the same command used in terminal, with class-name overrides for `Non-palm` and `Palm`.

In [ ]:
import subprocess
import sys

script_path = PROJECT_ROOT / "scripts" / "train_binary_gate.py"
cmd = [
    sys.executable,
    str(script_path),
    "--data-dir",
    str(DATA_DIR),
    "--output-dir",
    str(OUTPUT_DIR),
    "--non-palm-class",
    NON_PALM_CLASS,
    "--palm-class",
    PALM_CLASS,
]

print(" ".join(cmd))
# Uncomment to run training from notebook:
# subprocess.run(cmd, check=True, cwd=str(PROJECT_ROOT))

C:\Users\jeffy\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\python.exe scripts/train_binary_gate.py --data-dir C:\Users\jeffy\Documents\PSM\PalmDetector\BinaryGateSplit_80_20 --output-dir saved_models --non-palm-class Non-palm --palm-class Palm


## Section 5 - Verify Generated Artifacts

In [ ]:
search_dirs = [OUTPUT_DIR, PROJECT_ROOT / "models"]
printed = False

for folder in search_dirs:
    if not folder.exists():
        continue
    for p in sorted(folder.glob("palm_presence_binary*")):
        if p.is_file():
            printed = True
            print(f"{p.name:55} {p.stat().st_size:>10} bytes ({folder})")

if not printed:
    print("No binary gate artifacts found yet.")
    print("Run Cell 8 to train and export artifacts.")

## Section 6 - Read Recommended Threshold from Manifest

In [7]:
import json

search_dirs = [OUTPUT_DIR, PROJECT_ROOT / "models", Path.cwd() / "saved_models"]
manifest_map = {}

for folder in search_dirs:
    if not folder.exists():
        continue
    for p in folder.glob("palm_presence_binary_manifest_*.json"):
        manifest_map[str(p.resolve())] = p.resolve()

manifest_files = sorted(manifest_map.values(), key=lambda p: p.stat().st_mtime)

if not manifest_files:
    print("No manifest found in output directory.")
    print("Run Cell 8 first, then run this cell again.")
    print("Checked folders:")
    for folder in search_dirs:
        print(" -", folder.resolve())
else:
    latest_manifest = manifest_files[-1]
    with latest_manifest.open("r", encoding="utf-8") as f:
        manifest = json.load(f)

    print("Manifest:", latest_manifest)
    print("Recommended threshold:", manifest.get("recommended_threshold"))
    print("Default metrics:", manifest.get("validation_metrics", {}).get("default"))
    print("Recommended metrics:", manifest.get("validation_metrics", {}).get("recommended"))

Manifest: C:\Users\jeffy\Documents\PSM\Project Coding\PSM_PalmFruitRipenessClassificationSystem\saved_models\palm_presence_binary_manifest_20260414_003824.json
Recommended threshold: 0.31
Default metrics: {'threshold': 0.6, 'accuracy': 0.98, 'precision': 1.0, 'recall': 0.96, 'f1': 0.9795918367346939, 'tp': 192, 'tn': 200, 'fp': 0, 'fn': 8}
Recommended metrics: {'threshold': 0.31, 'accuracy': 0.985, 'precision': 0.9754901960784313, 'recall': 0.995, 'f1': 0.9851485148514851, 'tp': 199, 'tn': 195, 'fp': 5, 'fn': 1}


## Section 7 - Runtime Handoff

Use these values when starting API/CLI inference:

- `PALM_BINARY_MODEL_PATH=saved_models/palm_presence_binary.tflite`
- `PALM_BINARY_THRESHOLD=<recommended_threshold from manifest>`

Example (PowerShell):

```powershell
$env:PALM_BINARY_MODEL_PATH = "saved_models/palm_presence_binary.tflite"
$env:PALM_BINARY_THRESHOLD = "0.31"
python api/app.py
```